# Data loading and preprocessing

## Import libraries and inspect data

In [1]:
import numpy as np
import pandas as pd
import re
import html
import nltk
import pathlib
import sklearn
import csv
import os
import spacy
from nltk.corpus import stopwords
from collections import Counter
from sklearn.datasets import load_files
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer

In [ ]:
# loading data from original_datasets folder (book, movie, sports)
book_data = "original_datasets/book/all_kindle_review.csv"
movie_data = "original_datasets/movie/movie.csv"
sports_data = "original_datasets/sports/sports.csv"

# previewing the book data
df = pd.read_csv(book_data, usecols=["rating", "reviewText", "summary"])
df.head()

,rating,reviewText,summary
0,3,"Jace Rankin may be short, but he's nothing to ...",Entertaining But Average
1,5,Great short read. I didn't want to put it dow...,Terrific menage scenes!
2,3,I'll start by saying this is the first of four...,Snapdragon Alley
3,3,Aggie is Angela Lansbury who carries pocketboo...,very light murder cozy
4,4,I did not expect this type of book to be in li...,Book


## Data cleaning and adding to tsv

In [ ]:
# Load the spacy model
nlp = spacy.load("en_core_web_sm")
new_tsv = "sentiment_topic_processed.tsv"

# clean text in each of csv row
def clean_data(text):
    # replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)
    
    # remove links best I can :-(
    #text = re.sub(r'http[s]?://\S+|www\.\S+', '', text)
    text = re.sub(r'http[s]?://\S+|www\.\S+|\S+\.com|\S+\.net|\S+\.org|\S+\.co\S*', '', text)
    text = re.sub(r'\b(?:https|http|www)\s+\S+', '', text)
    text = re.sub(r'\S+\.(com|net|org|co\S*|io|tv|edu|gov|info)\b', '', text)
    text = re.sub(r'\S+\.(jpg|jpeg|png|gif|bmp|svg)\b', '', text)
    text = re.sub(r'\b(?:https?|www)\S*imgur\S*', '', text)
    
    # converting the html entity to their corresponding representation
    text = html.unescape(text)
    
    return (text)

# sentence splitting and start to put into dictionary
def split_to_sentences(text):
    cleaned_text = clean_data(text)
    processed_text = nlp(cleaned_text)
    split_sentences = [sent.text.strip() for sent in processed_text.sents]
    
    # to match the test set
    split_sentences = [sentence.capitalize() if sentence else sentence for sentence in split_sentences]
    return split_sentences

# adding data to training tsv
def add_to_tsv(input, output):

    data = []
    #row_count = 0
    
    def get_folder_name(file):
        return os.path.basename(os.path.dirname(file))
    
    def assign_sentiment(rating, topic):
        if topic == "book":
            if rating >= 4:
                return ("positive")
            elif rating == 3:
                return ("neutral")
            else:
                return ("negative")
        elif topic == "movie":
            if rating == 2:
                return ("positive")
            elif rating == 1:
                return ("neutral")
            else:
                return ("negative")
        else:
            return (rating)
    
    with open(input, "r", encoding="utf-8") as infile:
        reader = csv.DictReader(infile)
        
        for row in reader:
            #if row_count >= 15000: 
            #    break
            topic = get_folder_name(input)
            
            if topic == "book":
                review = row["reviewText"]
                rating = int(row["rating"])
            elif topic == "movie":
                review = row["review"]
                rating = int(row["sentiment"])
            elif topic == "sports":
                review = row["text"]
                rating = row["class_sentiment"]
            
            if not review or review.isspace():
                continue

            sentences = split_to_sentences(review)
            
            # only handle entries if there are 4 sentences or less
            if len(sentences) <= 4:
                sentiment = assign_sentiment(rating, topic)
                
                for sentence in sentences:
                    data.append({
                        "sentence": sentence,
                        "sentiment": sentiment,
                        "topic": topic
                    })
            #row_count += 1
        
    df = pd.DataFrame(data, columns=["sentence", "sentiment", "topic"])
    df.to_csv(output, sep="\t", index=True, index_label='sentence_id', mode="a", header=True)
    

#add_to_tsv(book_data, new_tsv)
#add_to_tsv(movie_data, new_tsv)
add_to_tsv(sports_data, new_tsv)

## Seeing how many neutral, positive and negative instances there are 

In [ ]:
# TSV file names
book_tsv = "training_sets/sentiment-topic/individual/book_processed.tsv"
movie_tsv = "training_sets/sentiment-topic/individual/movie_processed.tsv"
sports_tsv = "training_sets/sentiment-topic/individual/sports_processed.tsv"

def check_polarity_count(*tsvs):
    """
    Count the number of instances per polarity per topic.
    """
    for tsv in tsvs:
        neg_count = 0
        pos_count = 0
        neutral_count = 0
    
        with open(tsv, "r", encoding="utf-8") as infile:
            reader = csv.DictReader(infile, delimiter='\t')
            
            for row in reader:
                #print(row)
                if row["sentiment"] == "neutral":
                    neutral_count += 1
                elif row["sentiment"] == "positive":
                    pos_count += 1
                elif row["sentiment"] == "negative":
                    neg_count += 1
        
        dataset_name = os.path.basename(tsv).replace("_processed.tsv", "").upper()
    
        print(f"{dataset_name}: neutral {neutral_count}, negative {neg_count}, positive {pos_count}")

check_polarity_count(book_tsv, movie_tsv, sports_tsv)

BOOK: neutral 2683, negative 5431, positive 8473
MOVIE: neutral 526, negative 745, positive 430
SPORTS: neutral 6630, negative 3407, positive 7545


## Combine the tsvs from the 3 different topics

In [ ]:
# Output TSV file name
out = "combined.tsv"

def combine_tsvs(book, movie, sports, output):
    """
    Combine individual files into a single TSV file.
    """
    book = pd.read_csv(book, sep="\t")
    movie = pd.read_csv(movie, sep="\t")
    sports = pd.read_csv(sports, sep="\t")
    
    book = book.drop(columns=['sentence_id'], errors='ignore')
    movie = movie.drop(columns=['sentence_id'], errors='ignore')
    sports = sports.drop(columns=['sentence_id'], errors='ignore')
    
    combined_df = pd.concat([book, movie, sports], ignore_index=True)
    combined_df.to_csv(out, sep="\t", index=True, index_label='sentence_id', mode="w", header=True)
    
combine_tsvs(book_tsv, movie_tsv, sports_tsv, out)